In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
import datetime as dt
from bs4 import BeautifulSoup as bs
import requests
import json
import time
from tqdm import tqdm
import random
import math
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")   # 경고 메세지 끄기
from matplotlib import rcParams
from matplotlib.ticker import StrMethodFormatter

In [ ]:
# 한글 깨지는 현상 방지
# rcParams['font.family'] = 'Malgun Gothic'   # Windows OS일 경우
rcParams['font.family'] = 'AppleGothic'   # Mac OS일 경우
rcParams['axes.unicode_minus'] = False   # 한글 폰트 사용 시, 마이너스 글자가 깨지는 현상 방지

In [ ]:
pd.set_option('display.max_columns', None)  # 모든 칼럼 출력
# pd.set_option('display.max_rows', None)     # 모든 행 출력

In [ ]:
df = pd.read_pickle('(Panel)Challengers_df_v3.pkl')
df_rglr = pd.read_stata('kpk_challengers_panel_v1_240202_uc.dta')

In [ ]:
df_rglr2 = df_rglr[['User_ID','Challenge_ID','User_regularity']]

In [ ]:
df_merge = pd.merge(df, df_rglr2, how='left', left_on=['user_id', 'challenge_id'], right_on=['User_ID', 'Challenge_ID'])
df_merge = df_merge.drop(['User_ID', 'Challenge_ID'], axis=1)
df_merge.rename(columns = {'User_regularity' : 'user_regularity'}, inplace = True)
df_merge

In [ ]:
# 종속변수 (다음 챌린지에 추가로 거는 deposit 금액) 생성
def calculate_deposit_diff(group):
    
    # 다음 챌린지에 추가로 거는 deposit 금액 append할 리스트
    next_list = []
    diff_list = []
    
    if len(group)==1:
        diff_list.append(np.nan)
        next_list.append(np.nan)
        group['uc_deposit_diffNext'] = diff_list
        group['uc_deposit_next'] = next_list
        
    elif len(group)>1:
        challenge_startDate_list = group['challenge_startDate'].drop_duplicates(keep='first').tolist()
        challenge_startDate_list2 = challenge_startDate_list[:-1]
        
        for start in challenge_startDate_list2:
            end = challenge_startDate_list[challenge_startDate_list.index(start)+1]   # 다음 챌린지 날짜
            group_idx = group.index[(group['challenge_startDate'] == start) | (group['challenge_startDate'] == end)].tolist()
            idx_start = group_idx[0]
            idx_end = group_idx[-1]
            group_df = group.loc[idx_start:idx_end]   # 다음 챌린지에 추가로 거는 deposit 금액 계산할 데이터프레임
            for idx in group_idx:
                if group_df['challenge_startDate'][idx]==start:
                    next_df = group_df[group_df['challenge_startDate']==end]
                    next_deposit = next_df['uc_deposit'].mean()
                    deposit_diff = next_deposit - group_df['uc_deposit'][idx]
                    next_list.append(next_deposit)
                    diff_list.append(deposit_diff)
        # 마지막 날짜의 경우 계산이 불가능하므로 np.nan 부여
        cnt = len(group) - len(diff_list)
        for c in range(cnt):
            next_list.append(np.nan)
            diff_list.append(np.nan)
        # diff_list를 새로운 칼럼으로 생성
        group['uc_deposit_diffNext'] = diff_list
        group['uc_deposit_next'] = next_list
    return group

df2 = df_merge.groupby('user_id', group_keys=False).apply(calculate_deposit_diff)
df2

In [ ]:
# 해당 챌린지의 달성률 평균 칼럼 추가
df2['challenge_ach_rate_mean'] = df2.groupby('challenge_id')['uc_ach_rate_100'].transform('mean')
df2

In [ ]:
# 다음 챌린지의 챌린지 달성률 평균 칼럼 생성
def calculate_nextChallenge_achRate(group):
    
    # 다음 챌린지의 챌린지 달성률 평균 append할 리스트
    nextCAR_list = []
    
    if len(group)==1:
        nextCAR_list.append(np.nan)
        group['uc_nextChallenge_achRateMean'] = nextCAR_list
        
    elif len(group)>1:
        challenge_startDate_list = group['challenge_startDate'].drop_duplicates(keep='first').tolist()
        challenge_startDate_list2 = challenge_startDate_list[:-1]
        
        for start in challenge_startDate_list2:
            end = challenge_startDate_list[challenge_startDate_list.index(start)+1]   # 다음 챌린지 날짜
            group_idx = group.index[(group['challenge_startDate'] == start) | (group['challenge_startDate'] == end)].tolist()
            idx_start = group_idx[0]
            idx_end = group_idx[-1]
            group_df = group.loc[idx_start:idx_end]   # 다음 챌린지의 챌린지 달성률 평균 추가할 데이터프레임
            for idx in group_idx:
                if group_df['challenge_startDate'][idx]==start:
                    next_df = group_df[group_df['challenge_startDate']==end]
                    next_challengeAchRateMean = next_df['challenge_ach_rate_mean'].mean()
                    nextCAR_list.append(next_challengeAchRateMean)
        # 마지막 날짜의 경우 계산이 불가능하므로 np.nan 부여
        cnt = len(group) - len(nextCAR_list)
        for c in range(cnt):
            nextCAR_list.append(np.nan)
        # nextCAR_list를 새로운 칼럼으로 생성
        group['uc_nextChallenge_achRateMean'] = nextCAR_list
    return group

df3 = df2.groupby('user_id', group_keys=False).apply(calculate_nextChallenge_achRate)
df3

In [ ]:
# pickle로 저장
df3.to_pickle('(Panel)Challengers_df_v4.pkl')

---

In [ ]:
# dta 저장 시 에러가 날 가능성이 있는 칼럼 확인
string_columns = df3.select_dtypes(include=[object]).columns
df3[string_columns]

In [ ]:
cols_drop = ['goal_title', 'goal_category_level4', 'challenge_title']
df3 = df3.drop(cols_drop, axis=1)

In [ ]:
# dta 파일로 저장
df3.to_stata('Panel_Challengers_df_v4.dta', write_index=False)